# 26｜葡萄牙污水站点/批次差异、敏感性分析与论文图表整合

本笔记本完成葡萄牙污水真实案例的最后解释阶段。它只读取24号冻结预测和25号冻结后身份揭盲结果，不重新训练、不重新选择模型、不更改主危险性排序，也不会根据结果调整25号谱库主阈值。

分析包括：8个地点×3个采样批次的描述性比较、正负模式明细、固定谱库规则的阈值敏感性、Watch‑List CEC站点检出一致性、未被当前论文CEC/公共谱库命中的高危险候选，以及300 dpi论文图。所有“未知”均准确表述为**当前工作流未匹配候选**，不代表已经确认的新化合物。

In [ ]:
from pathlib import Path
import json, os, subprocess, time
import pandas as pd
from IPython.display import display, Image

ROOT = Path(r'E:\谱图预测PBT_终版')
ENV_MS = Path(r'D:\Appinstallation\Anaconda\envs\env_ms\python.exe')
SRC = ROOT / 'src'
CASE = ROOT / 'external_data' / 'portugal_wastewater_5830725'
MANIFEST_DIR = CASE / 'manifests'
OUT = CASE / '07_case_interpretation' / '26_portugal_case_interpretation'

RUN_ANALYSIS = True

assert ROOT.is_dir(), ROOT
assert ENV_MS.is_file(), ENV_MS
assert SRC.is_dir(), SRC
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC) + os.pathsep + env.get('PYTHONPATH', '')
env['PYTHONIOENCODING'] = 'utf-8'

print('env_ms:', ENV_MS)
print('output:', OUT)

## 1. 25号门控

必须确认25号为 `READY_FOR_NOTEBOOK_26_PORTUGAL_CASE_INTERPRETATION` 且全部必需QC通过。分析模块随后会逐个核对25号输出哈希；任何输入被改动都会立即停止。

In [ ]:
run25_path = MANIFEST_DIR / '25_identity_reveal_run_manifest.json'
assert run25_path.is_file(), run25_path
run25 = json.loads(run25_path.read_text(encoding='utf-8'))
assert run25['status'] == 'READY_FOR_NOTEBOOK_26_PORTUGAL_CASE_INTERPRETATION', run25['status']
assert run25['qc_all_required_pass'] is True
assert run25['identity_opened_only_after_notebook24_freeze'] is True

display(pd.Series({
    'Notebook 25 status': run25['status'],
    'Model-scored features': run25['counts']['model_scored_features'],
    'Common evaluation groups': run25['counts']['common_enrichment_universe_groups'],
    'Recovered published CECs': run25['counts']['recovered_unique_targets'],
    'Recovered Watch-List CECs': run25['counts']['recovered_watchlist_targets'],
}, name='value').to_frame())

## 2. 运行26号解释分析

该步骤主要使用CPU和内存，不需要GPU；通常几十秒内完成。谱库主规则保持25号预设的20 ppm、modified cosine≥0.70、匹配峰数≥6。敏感性网格只用于判断结论是否依赖阈值，不用于事后挑选更有利的身份结果。

In [ ]:
if RUN_ANALYSIS:
    cmd = [str(ENV_MS), '-m', 'pbt_msms.portugal_case_interpretation',
           '--root', str(ROOT)]
    print('Running:', ' '.join(cmd))
    started = time.perf_counter()
    subprocess.run(cmd, check=True, cwd=ROOT, env=env)
    print(f'Finished in {(time.perf_counter() - started) / 60:.2f} min')

run26_path = MANIFEST_DIR / '26_case_interpretation_run_manifest.json'
qc_path = MANIFEST_DIR / '26_case_interpretation_qc_checks.csv'
assert run26_path.is_file(), run26_path
run26 = json.loads(run26_path.read_text(encoding='utf-8'))
qc = pd.read_csv(qc_path)
assert run26['status'] == 'PORTUGAL_CASE_COMPLETE_READY_FOR_MANUSCRIPT_INTEGRATION', run26
assert run26['qc_all_required_pass'] is True
assert qc['passed'].astype(bool).all()

display(pd.Series(run26['counts'], name='count').to_frame())
display(qc)

## 3. 站点与采样批次模式

每个“地点×批次”合并正负模式，模式分层结果另存为独立表。主要站点指标是该样品单元中检出的、位于全案例冻结PBT危险性Top 5%的特征组数量，并按每1,000个完整端点特征标准化。

这些比较是**描述性和探索性**的：每个地点×批次×离子模式只有一个样品单元，不能把差异直接解释为因果污染梯度。Friedman检验仅比较同一地点三个批次的重复测量，p值未做多重校正，不作为主要确认性结论。

In [ ]:
site_campaign = pd.read_csv(OUT / '26_site_campaign_summary.csv')
site_summary = pd.read_csv(OUT / '26_site_summary.csv')
campaign_summary = pd.read_csv(OUT / '26_campaign_summary.csv')
friedman = pd.read_csv(OUT / '26_campaign_friedman_tests.csv')

display(site_campaign[[
    'site_code', 'campaign', 'eligible_common_group_n',
    'global_top_5pct_hazard_group_n', 'global_top_5pct_hazard_per_1000_eligible',
    'known_cec_target_n', 'watchlist_target_n'
]].sort_values(['site_code', 'campaign']).reset_index(drop=True))

display(site_summary[[
    'site_code', 'global_top_5pct_hazard_per_1000_eligible_mean',
    'global_top_5pct_hazard_per_1000_eligible_std',
    'known_cec_target_n_mean', 'watchlist_target_n_mean'
]].sort_values('global_top_5pct_hazard_per_1000_eligible_mean', ascending=False).reset_index(drop=True))

display(friedman)

## 4. 固定谱库规则的敏感性

网格覆盖10/15/20 ppm、cosine 0.60/0.70/0.80/0.85及至少4/6/8个匹配峰。红框对应25号主规则；无论其他阈值恢复多少目标，论文主身份结果仍使用红框规则。

In [ ]:
sensitivity = pd.read_csv(OUT / '26_library_threshold_sensitivity.csv')
primary_rule = sensitivity.loc[sensitivity['is_notebook25_primary_rule'].astype(bool)]
assert len(primary_rule) == 1
display(primary_rule)

display(sensitivity.loc[sensitivity['precursor_ppm_max'].eq(20.0), [
    'modified_cosine_min', 'matched_peaks_min', 'case_group_nonambiguous_n',
    'unique_target_n', 'watchlist_target_n', 'is_notebook25_primary_rule'
]].reset_index(drop=True))

## 5. 排序覆盖与未匹配高危险候选

主PBT排序和出现度辅助排序回答不同问题。前者强调预测危险性，后者把危险性与跨样品检出频率结合。已知CEC在出现度辅助排序中比例更高，是因为论文CEC本身偏向频繁检出物；这不是PBT模型性能优劣的直接证据。

未匹配候选可能包含真正未鉴定化合物，也可能包含公共谱库缺失物、不同加合物、同分异构体、原位碎片、共洗脱或特征对齐残差。因此只能作为后续鉴定优先清单。

In [ ]:
capture = pd.read_csv(OUT / '26_rank_capture_summary.csv')
candidates = pd.read_csv(OUT / '26_library_unmatched_high_priority_candidates.csv')

display(capture)
display(candidates['evidence_tier'].value_counts().rename_axis('tier').to_frame('candidate_n'))
display(candidates.head(30))

候选分层定义：

- **Tier A**：全案例PBT危险性Top 5%，且至少在2个样品中出现并跨≥2个地点或≥2个批次；
- **Tier B**：Top 5%但重复性较弱，或Top 10%且具重复检出；
- **Tier C**：其余Top 20%敏感性清单。

Tier是后续验证优先级，不是结构鉴定置信等级，也不是法规PBT判定。

## 6. 论文图表

主图依次展示：站点/批次高危险负荷、8个Watch‑List CEC的谱库检出一致性、谱库阈值敏感性、优先排序中的已知CEC注释比例。第二张图展示未匹配Top 20%候选在m/z–保留时间空间中的分布；青色边框为Tier A。

In [ ]:
display(Image(filename=str(OUT / 'figures' / '26_portugal_case_main_figure.png')))
display(Image(filename=str(OUT / 'figures' / '26_unmatched_high_priority_feature_map.png')))

## 7. 论文可用结论边界

可以支持的结论是：冻结直接谱图模型能为79,867个真实污水MS/MS特征提供危险性预测，而严格公共谱库仅覆盖其中630个样品级特征；因此直接模型显著扩大了可排序覆盖范围。主谱库规则恢复84/107个论文CEC及8/8个Watch‑List CEC，并且结论可通过阈值网格检查稳健性。站点与批次间高危险特征负荷存在描述性变化，同时仍有大量高危险候选未被当前谱库注释。

不能声称：所有未匹配候选都是新污染物；Level 2a命中是标准品确认；峰面积/频率等于浓度或风险；站点差异具有因果性；Watch‑List等同PBT真值。需要结构确证时，应从Tier A候选开始进行分子式、谱库扩展、保留时间和标准品验证。

In [ ]:
print('Notebook 26 final status:', run26['status'])
print('QC all required pass:', run26['qc_all_required_pass'])
print('Tier A unmatched candidates:', run26['counts']['tier_a_unmatched_candidates'])
print('Next step:', run26['next_step'])